# The risk-model axis: three covariance estimators and their conditioning

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.risk.covariance`

**Modules covered** `risk/covariance.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The risk layer carries the estimators the construction cells are built on: the sample covariance, linear shrinkage toward a structured target, and the covariance implied by the retained component model. The entry point prints each estimator's condition number and the shrinkage intensity it chose, which is how the axis is read.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `risk/covariance.py::conditioning` traces to the conditioning report of this effort, which is how the axis is read: at sixty months and eleven sleeves a sample covariance's condition number makes the optimiser the object under test far more than the estimator
- `risk/covariance.py::factor_model` traces to the factor-model covariance of this effort, reading the component structure the count rule retained: `B F B' + D`, which is why the risk layer depends on the factor layer
- `risk/covariance.py::main` traces to the risk-model axis of this effort: the sample covariance, linear shrinkage toward a structured target, and the factor-model covariance, each reported with the conditioning that decides which of them is usable at this ratio of series to observations
- `risk/covariance.py::sample` traces to the risk-model axis of this effort: the sample covariance, linear shrinkage toward a structured target, and the factor-model covariance, each reported with the conditioning that decides which of them is usable at this ratio of series to observations
- `risk/covariance.py::shrinkage` traces to linear shrinkage toward a structured target at the intensity the estimator itself estimates, Ledoit & Wolf (2004b), Journal of Multivariate Analysis 88(2), DOI 10.1016/S0047-259X(03)00096-4

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`risk/covariance.py`**

The risk-model axis: three estimators of one object behind one interface.

The comparison varies the covariance estimator with the constructor held fixed, and what that
axis measures is **the optimiser's numerical behaviour**, never which estimator is more
accurate. Nothing here estimates accuracy: there is no true covariance to compare against on
this panel, and a sample covariance is by construction the best fit *to its own window*, so an
in-sample accuracy ranking would rank the estimator it was computed from first.

Three choices are made once for all three estimators so that the axis compares estimators and
not conventions. They are all returned on the **correlation-free covariance scale in the
sleeve map's order**, with the labels travelling with the matrix. They all use the same
degrees-of-freedom convention: the shrinkage estimator is published maximum-likelihood scaled
and is rescaled here, because a comparison between an estimator that divides by T and one that
divides by T-1 measures the division rather than the estimator. And the factor covariance is
built on the correlation scale and de-standardised at the end, so the count that decides it is
the count the rule decided rather than one a volatile sleeve would have driven.

The reconstruction identity is checked rather than asserted: the residual variances left after
the retained components are the discarded eigenvalues, summed. That is arithmetic, not an
empirical claim, and a violation would mean the factor covariance is not the decomposition it
is described as.

This module reads the component model from `factors/`, so `risk/` depends on it: the PCA factor
covariance *is* the statistical family's covariance, and re-extracting the components here to
avoid the import would be the same model built twice, which is how two modules come to disagree
about what a component is. The layout's dependency arrows name `risk/ -> data`; the estimator
this axis actually varies needs `factors` as well.

## 3. The data contract it consumes, and the as-of rule

One matrix per window over the eleven sleeves in the universe's own order, so a matrix cannot silently disagree with the weight vectors it is used with. Every estimator is symmetric and positive semi-definite, and the intensity the shrinkage chose is returned beside the matrix rather than hidden inside it. The factor-model estimator reads the component structure the retention rule keeps, which is why the risk layer depends on the factor layer.

## 4. The worked example on small numbers, with the identity checked

A two-sleeve case with known variances: the sample estimator reproduces them exactly, shrinkage lands between the sample and its target, and shrinkage's condition number is the better of the two. The identity checked is that the estimators are the objects their names claim.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import numpy as np
import pandas as pd

from portfolio_workbench.risk import covariance

months = pd.PeriodIndex([f"2020-{i + 1:02d}" for i in range(12)], freq="M")
rng = np.random.default_rng(3)
frame = pd.DataFrame({"a": rng.normal(0.0, 0.02, 12), "b": rng.normal(0.0, 0.01, 12)}, index=months)

sample = covariance.sample(frame)
assert np.allclose(np.diag(sample.to_numpy()), np.var(frame, ddof=1), atol=1e-15)

report = covariance.conditioning(frame)
assert report["sample_condition"] > report["shrinkage_condition"]
assert 0.0 < report["intensity"] < 1.0
assert set(report["covariances"]) == {"sample", "shrinkage", "factor"}
print(f"sample condition {report['sample_condition']:.1f} against shrinkage {report['shrinkage_condition']:.1f}, "
      f"intensity {report['intensity']:.4f}")

sample condition 20.9 against shrinkage 2.8, intensity 0.4731


numpy/_core/fromnumeric.py:4230: FutureWarning: The behavior of DataFrame.var with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return var(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document["months"]
print(f"snapshot {document['snapshot_id']}, taken as of {document['as_of']}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document["manifest"])))

from portfolio_workbench.risk import covariance

print("the axis: the sample covariance, linear shrinkage, and the factor-model covariance")
print(f"reconstruction tolerance {covariance.RECONSTRUCTION_TOLERANCE:.0e} on the discarded eigenvalues")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
the axis: the sample covariance, linear shrinkage, and the factor-model covariance
reconstruction tolerance 1e-10 on the discarded eigenvalues


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.risk.covariance"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[risk] snapshot 2026-09-13, the window behind the last traded month 2026-07: 2021-07..2026-06, 60 observations × 11 sleeves
[risk] sample covariance: condition number 65,010
[risk] linear shrinkage: condition number 92.3 at intensity 0.070 - the intensity is how much of the sample the estimator judged to be noise, and it is a statement about the window rather than a defect
[risk] factor covariance: condition number 39,491.1 on 1 component(s), discarding 44.18% of the correlation variance (4.860000 against a residual sum of 4.860000)
[risk] all three carry the sleeve map's order and are symmetric; the axis measures the optimiser's numerical behaviour, and no estimator is ranked for accuracy, which this panel cannot establish
[risk] the three diagnostics this window produces for the estimator axis: sample conditioning 65,010, shrunk 92, factor 39,491
[risk] WARNING recomputed-TR divergence: XACT-NORDEN.ST adjusted close implies +424.74% cumulative, close plus distributions implies +640.2

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The condition number is what a constructor feels, and it is reported because at sixty months and eleven sleeves the sample covariance's condition number makes the optimiser the object under test far more than the estimator. Shrinkage's intensity is a result about the window: an intensity near one says the sample carried almost no information, which is not a defect of the estimator. No accuracy is claimed or measurable from these numbers, and a reader must not read a better-conditioned matrix as a more accurate one.

## 7. What this module does not establish

Nothing here establishes that any estimator forecasts future covariance better than another. Nothing establishes the target the shrinkage is drawn toward, which is a choice inside the estimator rather than a finding of this layer, and nothing here addresses tail dependence or regime change beyond the window's own variability.